# CVRP Heuristic Benchmarking on X Set

This notebook runs and analyzes heuristic solvers from `models/` on `sets/X/instances`.

What this covers:
- Dynamic model discovery from `models/`
- Batch benchmarking and cached JSON outputs under `outputs/`
- Quality and runtime metrics
- Gap against both:
  - best model found per instance (internal benchmark baseline)
  - best-known solution (BKS) from `sets/X/solutions/*.sol`
- Performance breakdowns by instance conditions (size, fleet density, demand tightness)

In [ ]:
# Core imports
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# Ensure project root is importable
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
# Benchmarking utilities from this repo
from benchmarking.run_benchmark import (
    discover_models,
    run_models,
    load_results,
    compute_metrics,
)
from benchmarking.load_cvrp import load_cvrp_dataset

In [ ]:
# Configuration
SETS_DIR = ROOT / "sets"
SET_NAME = "X"
MODELS_DIR = ROOT / "models"

# Keep notebook outputs under outputs/
OUTPUTS_ROOT = ROOT / "outputs"
RUN_ID = "x_heuristics_notebook"
BENCH_OUTPUT_DIR = OUTPUTS_ROOT / RUN_ID
PLOTS_DIR = BENCH_OUTPUT_DIR / "plots"

# Execution controls
OVERWRITE_EXISTING_RESULTS = False   # False => skip if JSON result already exists (cached)
STOP_ON_ERROR = False                # True => fail on first solver error
JOBS = 10                            # parallel solver threads (set 1 for serial)

# Instance selection controls
SMALL_MAX_CUSTOMERS = 250
INSTANCE_COUNT = 1                   # set None to use all small instances
RANDOM_INSTANCE_SELECTION = False     # True: random sample, False: first INSTANCE_COUNT
RANDOM_SEED = 42

BENCH_OUTPUT_DIR


In [4]:
# Load X-set instances, then select configurable number of small instances
full_x_dataset = load_cvrp_dataset(
    sets_dir=SETS_DIR,
    set_names=[SET_NAME],
    n_amount=None,
    random_selection=False,
    seed=RANDOM_SEED,
    verbose=False,
)


def _n_customers(problem):
    depots = list(getattr(problem, "depots", []) or [])
    depot_id = depots[0] if depots else 1
    return sum(1 for nid in problem.node_coords if nid != depot_id)


small_items = [
    (name, item)
    for name, item in full_x_dataset.items()
    if _n_customers(item["problem"]) <= SMALL_MAX_CUSTOMERS
]

small_items = sorted(small_items, key=lambda t: t[0])

if INSTANCE_COUNT is None:
    selected_pairs = small_items
else:
    if INSTANCE_COUNT < 1:
        raise ValueError("INSTANCE_COUNT must be >= 1 or None")
    if len(small_items) < INSTANCE_COUNT:
        raise ValueError(
            f"Requested {INSTANCE_COUNT} small instances, but only {len(small_items)} are available"
        )

    if RANDOM_INSTANCE_SELECTION:
        rng = np.random.default_rng(RANDOM_SEED)
        chosen_idx = rng.choice(len(small_items), size=INSTANCE_COUNT, replace=False)
        selected_pairs = [small_items[i] for i in sorted(chosen_idx)]
    else:
        selected_pairs = small_items[:INSTANCE_COUNT]

x_dataset = {name: item for name, item in selected_pairs}
instances = {name: item["problem"] for name, item in x_dataset.items()}

size_series = pd.Series(
    {name: _n_customers(problem) for name, problem in instances.items()},
    name="n_customers",
).sort_values()

print(f"Small-candidate pool: {len(small_items)}")
print(f"Selected instances: {len(instances)}")
print(f"Selection mode: {'random' if RANDOM_INSTANCE_SELECTION else 'first-N'}")
size_series


Small-candidate pool: 33
Selected instances: 1
Selection mode: first-N


X/X-n101-k25    100
Name: n_customers, dtype: int64

In [5]:
# Discover all runnable model heuristics in models/
models = discover_models(MODELS_DIR)

model_df = pd.DataFrame(
    {
        "model_name": [m.model_name for m in models],
        "source_file": [m.source_file.name for m in models],
        "function_name": [m.function_name for m in models],
    }
).sort_values(["source_file", "function_name"]).reset_index(drop=True)

print(f"Discovered model functions: {len(models)}")
model_df

# Remove extroardinaroy long run-time models
models = [
    m
    for m in models
    if m.model_name
    not in {
        "large-neighborhood-search__large_neighborhood_search",
        "tabu__tabu_search",
        "large-neighborhood-search__lns_from_nearest_neighbor",
    }
]

[discover] skipping improve-inter.py: no runnable solve-like function
[discover] skipping improve-intra.py: no runnable solve-like function
Discovered model functions: 18


In [6]:
# Run benchmark (cached by default if JSON already exists)
BENCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

run_df = run_models(
    models=models,
    instances=instances,  # dict: {instance_id: problem}
    outputs_dir=BENCH_OUTPUT_DIR,
    overwrite=OVERWRITE_EXISTING_RESULTS,
    stop_on_error=STOP_ON_ERROR,
    jobs=JOBS,
)

print(f"Parallel jobs: {JOBS}")
print(f"Newly executed rows in this run: {len(run_df)}")


Parallel jobs: 10
Newly executed rows in this run: 0


In [7]:
# Load full cached results and compute core metrics
results_df = load_results(BENCH_OUTPUT_DIR)
detailed_df, summary_df = compute_metrics(results_df)

print("Result rows:", len(results_df))
print("Detailed rows (successful only):", len(detailed_df))
print("Models in summary:", len(summary_df))

summary_df.head(10)

Result rows: 123
Detailed rows (successful only): 123
Models in summary: 18


,model_name,instances,mean_total_distance,median_total_distance,mean_runtime_sec,median_runtime_sec,mean_percent_gap,median_percent_gap,best_hits,attempted_runs,successful_runs,success_rate
0,memetic__memetic_from_nearest_neighbor,6,30628.094445,30294.842889,4343.940640,5294.068330,0.602841,0.583428,2,6,6,1.0
1,memetic__memetic_algorithm,6,30628.094445,30294.842889,5195.285621,6162.682208,0.602841,0.583428,2,6,6,1.0
2,iterated-local-search__ils_from_nearest_neighbor,6,30603.894574,30211.576820,848.150953,956.207604,0.690030,0.000000,4,6,6,1.0
3,iterated-local-search__iterated_local_search,6,30603.894574,30211.576820,851.128203,945.802531,0.690030,0.000000,4,6,6,1.0
4,constructive__clarke_wright,9,26786.910170,27275.765838,3.780313,2.840725,1.262011,1.247700,2,9,9,1.0
5,large-neighborhood-search__lns_from_nearest_ne...,5,31193.667981,32177.650946,2755.925256,2718.569465,1.271724,0.509007,0,5,5,1.0
6,large-neighborhood-search__large_neighborhood_...,5,31193.667981,32177.650946,2783.490403,2787.026487,1.271724,0.509007,0,5,5,1.0
7,genetic__ga_from_nearest_neighbor,7,30783.401177,30325.636980,76.950228,31.652117,2.641695,2.449206,1,7,7,1.0
8,genetic__genetic_algorithm,7,30783.401177,30325.636980,100.141146,27.855794,2.641695,2.449206,1,7,7,1.0
9,tabu__tabu_from_nearest_neighbor,6,33773.243082,32784.354258,38.716648,45.472921,11.026254,11.075621,0,6,6,1.0


In [8]:
# Parse best-known solution (BKS) costs from .sol files

def parse_bks_cost(sol_path):
    for line in sol_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line.lower().startswith("cost"):
            parts = line.split()
            try:
                return float(parts[-1])
            except Exception:
                return np.nan
    return np.nan

bks_records = []
for instance_id, item in x_dataset.items():
    sol_path = item.get("solution_path")
    bks_cost = np.nan
    if sol_path is not None and Path(sol_path).exists():
        bks_cost = parse_bks_cost(Path(sol_path))
    bks_records.append({"instance_id": instance_id, "bks_cost": bks_cost})

bks_df = pd.DataFrame(bks_records)
print("BKS coverage:", bks_df["bks_cost"].notna().mean())
bks_df.head()

BKS coverage: 1.0


,instance_id,bks_cost
0,X/X-n101-k25,27591.0


In [9]:
# Build instance-level condition features for segmented analysis

def build_instance_features(dataset):
    rows = []
    for instance_id, item in dataset.items():
        p = item["problem"]

        depots = list(getattr(p, "depots", []) or [])
        depot_id = depots[0] if depots else 1

        node_coords = getattr(p, "node_coords", {}) or {}
        demands = getattr(p, "demands", {}) or {}

        customer_ids = [nid for nid in sorted(node_coords.keys()) if nid != depot_id]
        n_customers = len(customer_ids)

        customer_demands = np.array([float(demands.get(cid, 0.0)) for cid in customer_ids], dtype=float)
        total_demand = float(customer_demands.sum())
        mean_demand = float(customer_demands.mean()) if n_customers else np.nan
        std_demand = float(customer_demands.std()) if n_customers else np.nan
        demand_cv = float(std_demand / mean_demand) if mean_demand and mean_demand > 0 else np.nan

        capacity = float(getattr(p, "capacity", np.nan))

        coords = np.array([node_coords[cid] for cid in customer_ids], dtype=float) if n_customers else np.empty((0, 2))
        if len(coords) > 0:
            x_span = float(coords[:, 0].max() - coords[:, 0].min())
            y_span = float(coords[:, 1].max() - coords[:, 1].min())
            bbox_area = float(max(1e-9, x_span * y_span))
            centroid = coords.mean(axis=0)
            centroid_dist_mean = float(np.linalg.norm(coords - centroid, axis=1).mean())
        else:
            x_span = np.nan
            y_span = np.nan
            bbox_area = np.nan
            centroid_dist_mean = np.nan

        m = re.search(r"-k(\d+)$", instance_id.split("/")[-1])
        k_from_name = float(m.group(1)) if m else np.nan

        demand_tightness = np.nan
        if np.isfinite(k_from_name) and k_from_name > 0 and np.isfinite(capacity) and capacity > 0:
            demand_tightness = total_demand / (k_from_name * capacity)

        k_per_100_customers = np.nan
        if np.isfinite(k_from_name) and n_customers > 0:
            k_per_100_customers = 100.0 * k_from_name / n_customers

        rows.append(
            {
                "instance_id": instance_id,
                "instance_name": instance_id.split("/")[-1],
                "n_customers": n_customers,
                "capacity": capacity,
                "total_demand": total_demand,
                "mean_demand": mean_demand,
                "std_demand": std_demand,
                "demand_cv": demand_cv,
                "x_span": x_span,
                "y_span": y_span,
                "bbox_area": bbox_area,
                "centroid_dist_mean": centroid_dist_mean,
                "k_from_name": k_from_name,
                "demand_tightness": demand_tightness,
                "k_per_100_customers": k_per_100_customers,
            }
        )

    feat = pd.DataFrame(rows)

    # Condition buckets
    feat["size_bucket"] = pd.cut(
        feat["n_customers"],
        bins=[0, 250, 500, 750, np.inf],
        labels=["small(<=250)", "medium(251-500)", "large(501-750)", "xlarge(>750)"],
    )

    feat["fleet_density_bucket"] = pd.cut(
        feat["k_per_100_customers"],
        bins=[-np.inf, 8, 16, np.inf],
        labels=["low", "medium", "high"],
    )

    feat["tightness_bucket"] = pd.cut(
        feat["demand_tightness"],
        bins=[-np.inf, 0.75, 0.90, np.inf],
        labels=["loose", "moderate", "tight"],
    )

    return feat


instance_features_df = build_instance_features(x_dataset)
instance_features_df.head()

,instance_id,instance_name,n_customers,capacity,total_demand,mean_demand,std_demand,demand_cv,x_span,y_span,bbox_area,centroid_dist_mean,k_from_name,demand_tightness,k_per_100_customers,size_bucket,fleet_density_bucket,tightness_bucket
0,X/X-n101-k25,X-n101-k25,100,206.0,5147.0,51.47,28.712525,0.55785,965.0,986.0,951490.0,368.959716,25.0,0.999417,25.0,small(<=250),high,tight


In [10]:
# Merge benchmark metrics with BKS and condition features
analysis_df = detailed_df.merge(instance_features_df, on="instance_id", how="left")
analysis_df = analysis_df.merge(bks_df, on="instance_id", how="left")

analysis_df["bks_gap_pct"] = (
    (analysis_df["total_distance"] - analysis_df["bks_cost"])
    / analysis_df["bks_cost"]
) * 100.0
analysis_df.loc[analysis_df["bks_cost"].isna(), "bks_gap_pct"] = np.nan

analysis_df["is_best_internal"] = (analysis_df["percent_gap"].abs() <= 1e-9).astype(int)

print("Rows in analysis_df:", len(analysis_df))
analysis_df.head()

Rows in analysis_df: 123


,status,error,model_name,module_name,function_name,source_file,instance_id,started_at_utc,runtime_sec,reported_runtime_sec,total_distance,routes,raw_output,result_file,best_total_distance,percent_gap,instance_name,n_customers,capacity,total_demand,mean_demand,std_demand,demand_cv,x_span,y_span,bbox_area,centroid_dist_mean,k_from_name,demand_tightness,k_per_100_customers,size_bucket,fleet_density_bucket,tightness_bucket,bks_cost,bks_gap_pct,is_best_internal
0,ok,None,ant-colony__aco_from_nearest_neighbor,benchmark_dynamic_ant_colony_0,aco_from_nearest_neighbor,/home/manpazito/projects/cvrp-ml-tradeoffs/mod...,X/X-n101-k25,2026-03-11T22:56:49.114507+00:00,6.413444,NaN,36146.502114,"[[33, 47, 25, 32, 74, 65], [27, 49, 29, 40, 66...","{'best_routes': [[33, 47, 25, 32, 74, 65], [27...",/home/manpazito/projects/cvrp-ml-tradeoffs/out...,28368.954453,27.415701,X-n101-k25,100.0,206.0,5147.0,51.47,28.712525,0.55785,965.0,986.0,951490.0,368.959716,25.0,0.999417,25.0,small(<=250),high,tight,27591.0,31.008307,0
1,ok,None,ant-colony__aco_from_nearest_neighbor,benchmark_dynamic_ant_colony_0,aco_from_nearest_neighbor,/home/manpazito/projects/cvrp-ml-tradeoffs/mod...,X/X-n106-k14,2026-03-11T22:56:49.126600+00:00,9.750248,NaN,28901.488229,"[[23, 34, 53, 27, 63, 91, 103], [97, 46, 101, ...","{'best_routes': [[23, 34, 53, 27, 63, 91, 103]...",/home/manpazito/projects/cvrp-ml-tradeoffs/out...,27274.845202,5.963895,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,ok,None,ant-colony__aco_from_nearest_neighbor,benchmark_dynamic_ant_colony_0,aco_from_nearest_neighbor,/home/manpazito/projects/cvrp-ml-tradeoffs/mod...,X/X-n110-k13,2026-03-11T22:36:03.645483+00:00,5.764539,NaN,19132.881380,"[[50, 42, 13, 69, 55, 51, 53, 40, 32], [15, 10...","{'best_routes': [[50, 42, 13, 69, 55, 51, 53, ...",/home/manpazito/projects/cvrp-ml-tradeoffs/out...,15485.723199,23.551746,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,ok,None,ant-colony__aco_from_nearest_neighbor,benchmark_dynamic_ant_colony_0,aco_from_nearest_neighbor,/home/manpazito/projects/cvrp-ml-tradeoffs/mod...,X/X-n115-k10,2026-03-11T22:56:49.127395+00:00,11.110273,NaN,18261.818055,"[[16, 33, 90, 18, 67, 17, 105, 42, 92, 3, 24, ...","{'best_routes': [[16, 33, 90, 18, 67, 17, 105,...",/home/manpazito/projects/cvrp-ml-tradeoffs/out...,13493.321851,35.339676,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,ok,None,ant-colony__aco_from_nearest_neighbor,benchmark_dynamic_ant_colony_0,aco_from_nearest_neighbor,/home/manpazito/projects/cvrp-ml-tradeoffs/mod...,X/X-n120-k6,2026-03-11T22:56:49.138090+00:00,13.041271,NaN,15848.901356,"[[21, 117, 108, 62, 120, 95, 11, 89, 53, 51, 3...","{'best_routes': [[21, 117, 108, 62, 120, 95, 1...",/home/manpazito/projects/cvrp-ml-tradeoffs/out...,14539.780146,9.003721,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [11]:
# Overall model ranking table (internal best + BKS)
overall = (
    analysis_df.groupby("model_name", as_index=False)
    .agg(
        instances=("instance_id", "nunique"),
        mean_distance=("total_distance", "mean"),
        mean_runtime_sec=("runtime_sec", "mean"),
        median_runtime_sec=("runtime_sec", "median"),
        mean_internal_gap_pct=("percent_gap", "mean"),
        median_internal_gap_pct=("percent_gap", "median"),
        internal_wins=("is_best_internal", "sum"),
        mean_bks_gap_pct=("bks_gap_pct", "mean"),
        median_bks_gap_pct=("bks_gap_pct", "median"),
    )
)

overall["internal_win_rate"] = overall["internal_wins"] / overall["instances"].clip(lower=1)
overall = overall.sort_values(["mean_bks_gap_pct", "mean_internal_gap_pct", "mean_runtime_sec"], ascending=[True, True, True])

overall

,model_name,instances,mean_distance,mean_runtime_sec,median_runtime_sec,mean_internal_gap_pct,median_internal_gap_pct,internal_wins,mean_bks_gap_pct,median_bks_gap_pct,internal_win_rate
8,iterated-local-search__ils_from_nearest_neighbor,6,30603.894574,848.150953,956.207604,0.690030,0.000000,4,2.819595,2.819595,0.666667
9,iterated-local-search__iterated_local_search,6,30603.894574,851.128203,945.802531,0.690030,0.000000,4,2.819595,2.819595,0.666667
13,memetic__memetic_from_nearest_neighbor,6,30628.094445,4343.940640,5294.068330,0.602841,0.583428,2,3.566351,3.566351,0.333333
12,memetic__memetic_algorithm,6,30628.094445,5195.285621,6162.682208,0.602841,0.583428,2,3.566351,3.566351,0.333333
3,constructive__clarke_wright,9,26786.910170,3.780313,2.840725,1.262011,1.247700,2,4.896468,4.896468,0.222222
6,genetic__ga_from_nearest_neighbor,7,30783.401177,76.950228,31.652117,2.641695,2.449206,1,9.911337,9.911337,0.142857
7,genetic__genetic_algorithm,7,30783.401177,100.141146,27.855794,2.641695,2.449206,1,9.911337,9.911337,0.142857
16,tabu__tabu_from_nearest_neighbor,6,33773.243082,38.716648,45.472921,11.026254,11.075621,0,12.403231,12.403231,0.000000
1,ant-colony__ant_colony_optimization,9,31366.388486,21.712116,12.837388,18.869053,18.085188,0,31.008307,31.008307,0.000000
0,ant-colony__aco_from_nearest_neighbor,9,31366.388486,21.866723,13.041271,18.869053,18.085188,0,31.008307,31.008307,0.000000


In [12]:
# Reliability / error profile from raw results
error_profile = (
    results_df.assign(is_ok=results_df["status"].eq("ok").astype(int))
    .groupby("model_name", as_index=False)
    .agg(
        attempted=("status", "size"),
        successful=("is_ok", "sum"),
    )
)
error_profile["failed"] = error_profile["attempted"] - error_profile["successful"]
error_profile["failure_rate"] = error_profile["failed"] / error_profile["attempted"].clip(lower=1)
error_profile.sort_values("failure_rate", ascending=False)

,model_name,attempted,successful,failed,failure_rate
0,ant-colony__aco_from_nearest_neighbor,9,9,0,0.0
1,ant-colony__ant_colony_optimization,9,9,0,0.0
2,constructive__cheapest_insertion,7,7,0,0.0
3,constructive__clarke_wright,9,9,0,0.0
4,constructive__nearest_neighbor,9,9,0,0.0
5,constructive__sweep,9,9,0,0.0
6,genetic__ga_from_nearest_neighbor,7,7,0,0.0
7,genetic__genetic_algorithm,7,7,0,0.0
8,iterated-local-search__ils_from_nearest_neighbor,6,6,0,0.0
9,iterated-local-search__iterated_local_search,6,6,0,0.0


In [13]:
# Family-level aggregation (e.g., constructive, tabu, genetic)
analysis_df["model_family"] = analysis_df["model_name"].str.split("__").str[0]

family_summary = (
    analysis_df.groupby("model_family", as_index=False)
    .agg(
        models=("model_name", "nunique"),
        mean_internal_gap_pct=("percent_gap", "mean"),
        mean_bks_gap_pct=("bks_gap_pct", "mean"),
        mean_runtime_sec=("runtime_sec", "mean"),
        win_rate=("is_best_internal", "mean"),
    )
    .sort_values(["mean_bks_gap_pct", "mean_internal_gap_pct", "mean_runtime_sec"], ascending=[True, True, True])
)

family_summary

,model_family,models,mean_internal_gap_pct,mean_bks_gap_pct,mean_runtime_sec,win_rate
3,iterated-local-search,2,0.690030,2.819595,849.639578,0.666667
5,memetic,2,0.602841,3.566351,4769.613130,0.333333
2,genetic,2,2.641695,9.911337,88.545687,0.142857
7,tabu,2,11.181292,12.403231,42.496676,0.000000
0,ant-colony,2,18.869053,31.008307,21.789420,0.000000
1,constructive,4,38.079761,34.340266,16.965257,0.058824
6,simulated-annealing,2,24.705917,50.484020,11.096821,0.000000
4,large-neighborhood-search,2,1.271724,NaN,2769.707830,0.000000


In [14]:
# Plot: family-level quality/runtime profile
plt.figure(figsize=(10, 6))
ax = sns.scatterplot(
    data=family_summary,
    x="mean_runtime_sec",
    y="mean_internal_gap_pct",
    hue="model_family",
    s=220,
)
ax.set_xscale("log")
ax.set_xlabel("Mean Runtime (sec, log scale)")
ax.set_ylabel("Mean Internal Gap (%)")
ax.set_title("Model Family Trade-off")
for _, row in family_summary.iterrows():
    ax.text(row["mean_runtime_sec"], row["mean_internal_gap_pct"], row["model_family"], fontsize=10)
plt.legend([], [], frameon=False)
plt.tight_layout()
plt.savefig("../outputs/x_heuristics_notebook/plots/family_quality_runtime_tradeoff.png")
plt.show()

/tmp/ipykernel_33927/2264556794.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Per-instance ranking stability (lower mean rank is better)
rank_df = analysis_df.copy()
rank_df["quality_rank"] = rank_df.groupby("instance_id")["total_distance"].rank(method="average", ascending=True)
rank_df["runtime_rank"] = rank_df.groupby("instance_id")["runtime_sec"].rank(method="average", ascending=True)

rank_summary = (
    rank_df.groupby("model_name", as_index=False)
    .agg(
        mean_quality_rank=("quality_rank", "mean"),
        median_quality_rank=("quality_rank", "median"),
        mean_runtime_rank=("runtime_rank", "mean"),
        median_runtime_rank=("runtime_rank", "median"),
    )
    .sort_values(["mean_quality_rank", "mean_runtime_rank"], ascending=[True, True])
)

rank_summary

,model_name,mean_quality_rank,median_quality_rank,mean_runtime_rank,median_runtime_rank
8,iterated-local-search__ils_from_nearest_neighbor,2.666667,1.5,13.333333,13.0
9,iterated-local-search__iterated_local_search,2.666667,1.5,13.333333,13.5
13,memetic__memetic_from_nearest_neighbor,3.666667,4.0,16.666667,17.0
12,memetic__memetic_algorithm,3.666667,4.0,17.333333,18.0
3,constructive__clarke_wright,4.666667,5.0,3.333333,3.0
11,large-neighborhood-search__lns_from_nearest_ne...,5.500000,5.5,15.200000,15.0
10,large-neighborhood-search__large_neighborhood_...,5.500000,5.5,15.800000,16.0
6,genetic__ga_from_nearest_neighbor,6.285714,8.0,9.285714,10.0
7,genetic__genetic_algorithm,6.285714,8.0,9.428571,9.0
16,tabu__tabu_from_nearest_neighbor,10.250000,10.5,9.500000,10.0


In [16]:
plot_df = overall.dropna(subset=["mean_runtime_sec", "mean_internal_gap_pct"]).copy()

plt.figure(figsize=(12, 8))

<Figure size 1200x800 with 0 Axes>

In [17]:
# Plot 1: Mean runtime vs mean quality gap (Pareto-style)
plot_df = overall.dropna(subset=["mean_runtime_sec", "mean_internal_gap_pct"]).copy()

plt.figure(figsize=(12, 8))
ax = sns.scatterplot(
    data=plot_df,
    x="mean_runtime_sec",
    y="mean_internal_gap_pct",
    hue="model_name",
    s=140,
)
ax.set_xscale("log")
ax.set_xlabel("Mean Runtime (sec, log scale)")
ax.set_ylabel("Mean Internal Gap (%)")
ax.set_title("Quality vs Runtime Trade-off Across Heuristics")

for _, row in plot_df.iterrows():
    ax.text(row["mean_runtime_sec"], row["mean_internal_gap_pct"], row["model_name"], fontsize=9)
    
    
plt.legend([], [], frameon=False)
plt.tight_layout()
plt.savefig("../outputs/x_heuristics_notebook/plots/quality_runtime_tradeoff.png")
plt.show()

/tmp/ipykernel_33927/3581208085.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Plot 2: Internal gap distribution by model
order = overall["model_name"].tolist()

plt.figure(figsize=(max(14, len(order) * 0.8), 7))
sns.boxplot(data=analysis_df, x="model_name", y="percent_gap", order=order)
plt.xticks(rotation=60, ha="right")
plt.xlabel("Model")
plt.ylabel("Percent Gap vs Best Model on Same Instance")
plt.title("Solution Quality Distribution by Model")
plt.tight_layout()
plt.savefig("../outputs/x_heuristics_notebook/plots/quality_distribution_by_model.png")
plt.show()

/tmp/ipykernel_33927/3625234918.py:10: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/tmp/ipykernel_33927/3625234918.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# Plot 3: Runtime distribution by model
order = overall["model_name"].tolist()

plt.figure(figsize=(max(14, len(order) * 0.8), 7))
sns.boxplot(data=analysis_df, x="model_name", y="runtime_sec", order=order)
plt.yscale("log")
plt.xticks(rotation=60, ha="right")
plt.xlabel("Model")
plt.ylabel("Runtime (sec, log scale)")
plt.title("Runtime Distribution by Model")
plt.tight_layout()
plt.savefig("../outputs/x_heuristics_notebook/plots/runtime_distribution_by_model.png")
plt.show()

/tmp/ipykernel_33927/2947941621.py:11: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/tmp/ipykernel_33927/2947941621.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# Condition analysis helper
def condition_breakdown(df, condition_col, metric_col="percent_gap"):
    out = (
        df.dropna(subset=[condition_col])
        .groupby([condition_col, "model_name"], as_index=False)
        .agg(
            instances=("instance_id", "nunique"),
            mean_metric=(metric_col, "mean"),
            median_metric=(metric_col, "median"),
            mean_runtime_sec=("runtime_sec", "mean"),
            win_rate=("is_best_internal", "mean"),
        )
    )
    return out

In [21]:
# Plot 4: Heatmap of mean internal gap by size bucket
size_perf = condition_breakdown(analysis_df, "size_bucket", metric_col="percent_gap")
size_heat = size_perf.pivot(index="model_name", columns="size_bucket", values="mean_metric")
size_heat = size_heat.reindex([m for m in overall["model_name"] if m in size_heat.index])

plt.figure(figsize=(10, max(7, len(size_heat) * 0.35)))
sns.heatmap(size_heat, cmap="viridis", annot=False)
plt.title("Mean Internal Gap (%) by Model and Instance Size Bucket")
plt.xlabel("Size Bucket")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig("../outputs/x_heuristics_notebook/plots/quality_by_size_heatmap.png")
plt.show()

size_perf.sort_values(["size_bucket", "mean_metric", "mean_runtime_sec"]).head(20)

/tmp/ipykernel_33927/1715254392.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.dropna(subset=[condition_col])


ValueError: Length of values (15) does not match length of index (60)

In [ ]:
# Plot 5: Heatmap of mean BKS gap by tightness bucket
bks_tight = condition_breakdown(analysis_df.dropna(subset=["bks_gap_pct"]), "tightness_bucket", metric_col="bks_gap_pct")
bks_tight_heat = bks_tight.pivot(index="model_name", columns="tightness_bucket", values="mean_metric")
bks_tight_heat = bks_tight_heat.loc[overall["model_name"]]

plt.figure(figsize=(9, max(7, len(bks_tight_heat) * 0.35)))
sns.heatmap(bks_tight_heat, cmap="magma", annot=False)
plt.title("Mean BKS Gap (%) by Model and Demand Tightness Bucket")
plt.xlabel("Demand Tightness Bucket")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig("../outputs/x_heuristics_notebook/plots/bks_gap_by_tightness_heatmap.png")
plt.show()

bks_tight.sort_values(["tightness_bucket", "mean_metric", "mean_runtime_sec"]).head(20)

In [ ]:
# Best model per condition (quality-first, then runtime)

def best_by_condition(df, condition_col, metric_col):
    tmp = (
        df.dropna(subset=[condition_col])
        .groupby([condition_col, "model_name"], as_index=False)
        .agg(
            instances=("instance_id", "nunique"),
            mean_metric=(metric_col, "mean"),
            mean_runtime_sec=("runtime_sec", "mean"),
        )
    )
    tmp = tmp.sort_values([condition_col, "mean_metric", "mean_runtime_sec"], ascending=[True, True, True])
    best = tmp.groupby(condition_col, as_index=False).head(1)
    return best

best_size = best_by_condition(analysis_df, "size_bucket", "percent_gap")
best_tight = best_by_condition(analysis_df.dropna(subset=["bks_gap_pct"]), "tightness_bucket", "bks_gap_pct")
best_fleet = best_by_condition(analysis_df, "fleet_density_bucket", "percent_gap")

print("Best by size bucket")
display(best_size)

print("Best by demand tightness bucket (BKS gap)")
display(best_tight)

print("Best by fleet density bucket")
display(best_fleet)

In [ ]:
# Save notebook-generated summary artifacts under outputs/
summary_path = BENCH_OUTPUT_DIR / "overall_model_summary.csv"
condition_size_path = BENCH_OUTPUT_DIR / "condition_size_summary.csv"
condition_tightness_path = BENCH_OUTPUT_DIR / "condition_tightness_bks_summary.csv"
error_path = BENCH_OUTPUT_DIR / "model_reliability.csv"

overall.to_csv(summary_path, index=False)
size_perf.to_csv(condition_size_path, index=False)
bks_tight.to_csv(condition_tightness_path, index=False)
error_profile.to_csv(error_path, index=False)

print("Saved:")
print("-", summary_path)
print("-", condition_size_path)
print("-", condition_tightness_path)
print("-", error_path)

## Notes and Interpretation Guidance

- `mean_internal_gap_pct` compares each model to the best model result on each instance in this benchmark run.
- `mean_bks_gap_pct` compares to best-known solution (from `.sol`) and is the stronger external quality metric.
- For production use, prioritize models that are consistently low in both `mean_bks_gap_pct` and `mean_runtime_sec`.
- Use condition tables (`size_bucket`, `tightness_bucket`, `fleet_density_bucket`) to pick different models for different instance regimes.